# Model 2: Refund Risk Probability
Predict whether an ordered item will be refunded (binary classification).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import joblib

# Uploading the required files to colab storage

In [ ]:
from google.colab import files
files.upload()

Saving order_item_refunds.csv to order_item_refunds.csv
Saving order_items.csv to order_items.csv
Saving orders.csv to orders.csv
Saving products.csv to products.csv


{'order_item_refunds.csv': b'order_item_refund_id,created_at,order_item_id,order_id,refund_amount_usd\n1,"2012-04-06 11:32:43",57,57,49.99\n2,"2012-04-13 01:09:43",74,74,49.99\n3,"2012-04-15 07:03:48",71,71,49.99\n4,"2012-04-17 20:00:37",118,118,49.99\n5,"2012-04-22 20:53:49",116,116,49.99\n6,"2012-05-04 11:59:07",147,147,49.99\n7,"2012-05-12 02:41:14",186,186,49.99\n8,"2012-05-16 13:06:01",191,191,49.99\n9,"2012-05-24 16:00:09",179,179,49.99\n10,"2012-05-30 17:20:44",199,199,49.99\n11,"2012-06-06 14:22:14",271,271,49.99\n12,"2012-06-10 19:54:47",290,290,49.99\n13,"2012-06-20 19:13:12",335,335,49.99\n14,"2012-06-28 18:54:22",382,382,49.99\n15,"2012-06-29 15:50:01",357,357,49.99\n16,"2012-07-05 14:24:53",409,409,49.99\n17,"2012-07-07 07:36:44",368,368,49.99\n18,"2012-07-07 19:54:28",391,391,49.99\n19,"2012-07-08 02:59:09",424,424,49.99\n20,"2012-07-10 14:31:05",393,393,49.99\n21,"2012-07-11 22:16:10",442,442,49.99\n22,"2012-07-19 06:57:16",472,472,49.99\n23,"2012-07-20 13:51:53",470,470

# Load raw data

In [ ]:
order_items = pd.read_csv('order_items.csv', parse_dates=['created_at'])
order_item_refunds = pd.read_csv('order_item_refunds.csv', parse_dates=['created_at'])
products = pd.read_csv('products.csv')
orders = pd.read_csv('orders.csv', parse_dates=['created_at'])

In [ ]:
order_items.head(3)

,order_item_id,created_at,order_id,product_id,is_primary_item,price_usd,cogs_usd
0,1,2012-03-19 10:42:46,1,1,1,49.99,19.49
1,2,2012-03-19 19:27:37,2,1,1,49.99,19.49
2,3,2012-03-20 06:44:45,3,1,1,49.99,19.49


In [ ]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40025 entries, 0 to 40024
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_item_id    40025 non-null  int64         
 1   created_at       40025 non-null  datetime64[ns]
 2   order_id         40025 non-null  int64         
 3   product_id       40025 non-null  int64         
 4   is_primary_item  40025 non-null  int64         
 5   price_usd        40025 non-null  float64       
 6   cogs_usd         40025 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(4)
memory usage: 2.1 MB


In [ ]:
order_item_refunds.head(3)

,order_item_refund_id,created_at,order_item_id,order_id,refund_amount_usd
0,1,2012-04-06 11:32:43,57,57,49.99
1,2,2012-04-13 01:09:43,74,74,49.99
2,3,2012-04-15 07:03:48,71,71,49.99


In [ ]:
order_item_refunds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1731 entries, 0 to 1730
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   order_item_refund_id  1731 non-null   int64         
 1   created_at            1731 non-null   datetime64[ns]
 2   order_item_id         1731 non-null   int64         
 3   order_id              1731 non-null   int64         
 4   refund_amount_usd     1731 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(3)
memory usage: 67.7 KB


# Data cleaning

In [ ]:
order_items.isnull().sum()

,0
order_item_id,0
created_at,0
order_id,0
product_id,0
is_primary_item,0
price_usd,0
cogs_usd,0


In [ ]:
order_items.duplicated().sum()

np.int64(0)

In [ ]:
order_item_refunds.isnull().sum()

,0
order_item_refund_id,0
created_at,0
order_item_id,0
order_id,0
refund_amount_usd,0


In [ ]:
order_item_refunds.duplicated().sum()

np.int64(0)

# Build the target variable
An order item is "refunded" if it has at least one matching row in order_item_refunds.
We dedupe refunds to 1 row per order_item_id before merging, to avoid fan-out.

In [ ]:
refunds_agg = order_item_refunds.groupby('order_item_id').size().reset_index(name='num_refund_records')

In [ ]:
data = order_items.merge(refunds_agg, on='order_item_id', how='left')

In [ ]:
data['num_refund_records'] = data['num_refund_records'].fillna(0)

In [ ]:
data['refunded'] = np.where(data['num_refund_records'] > 0, 1, 0)

In [ ]:
data['refunded'].value_counts()

,count
refunded,
0,38294
1,1731


In [ ]:
data['refunded'].value_counts(normalize=True)

,proportion
refunded,
0,0.956752
1,0.043248


## Class imbalance -- addressed explicitly (same as Model 1)
Refund rate is typically low. We handle this via class_weight='balanced' /
scale_pos_weight, stratified train/test split, and threshold tuning below --
not just accuracy, since a "never refund" model would score misleadingly high.

In [ ]:
data = data.merge(products, on='product_id', how='left')

In [ ]:
data = data.merge(orders[['order_id', 'total_amt_usd' if 'total_amt_usd' in orders.columns else 'price_usd']],
                   on='order_id', how='left', suffixes=('', '_order'))

In [ ]:
data.head(3)

,order_item_id,created_at_x,order_id,product_id,is_primary_item,price_usd,cogs_usd,num_refund_records,refunded,created_at_y,product_name,price_usd_order
0,1,2012-03-19 10:42:46,1,1,1,49.99,19.49,0.0,0,2012-03-19 08:00:00,The Original Mr. Fuzzy,49.99
1,2,2012-03-19 19:27:37,2,1,1,49.99,19.49,0.0,0,2012-03-19 08:00:00,The Original Mr. Fuzzy,49.99
2,3,2012-03-20 06:44:45,3,1,1,49.99,19.49,0.0,0,2012-03-19 08:00:00,The Original Mr. Fuzzy,49.99


In [ ]:
data['month'] = data['created_at_x'].dt.month
data['dayofweek'] = data['created_at_x'].dt.dayofweek

In [ ]:
data.columns

Index(['order_item_id', 'created_at_x', 'order_id', 'product_id',
       'is_primary_item', 'price_usd', 'cogs_usd', 'num_refund_records',
       'refunded', 'created_at_y', 'product_name', 'price_usd_order', 'month',
       'dayofweek'],
      dtype='object')

In [ ]:
feature_cols = ['is_primary_item', 'price_usd', 'cogs_usd',
        'product_id', 'price_usd_order', 'month',
       'dayofweek']
target = ['refunded']

In [ ]:
X = data[['is_primary_item', 'price_usd', 'cogs_usd',
        'product_id', 'price_usd_order', 'month',
       'dayofweek']]

In [ ]:
data.columns

Index(['order_item_id', 'created_at_x', 'order_id', 'product_id',
       'is_primary_item', 'price_usd', 'cogs_usd', 'num_refund_records',
       'refunded', 'created_at_y', 'product_name', 'price_usd_order', 'month',
       'dayofweek'],
      dtype='object')

In [ ]:
y = data['refunded']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123, stratify=y)

# Encode features for sklearn models

In [ ]:
X_train.shape, X_test.shape

((32020, 7), (8005, 7))

# Class imbalance ratio (for XGBoost's scale_pos_weight)

In [ ]:
neg, pos = np.bincount(y_train)
imbalance_ratio = neg / pos


# Model comparison

# Model A: Logistic Regression (class_weight='balanced')

In [ ]:
par_grid_lr = {'C': [0.01, 0.1, 1]}
grid_lr = GridSearchCV(LogisticRegression(max_iter=500, class_weight='balanced'),
                        param_grid=par_grid_lr, cv=3, scoring='roc_auc', n_jobs=-1)
grid_lr = grid_lr.fit(X_train, y_train)

In [ ]:
grid_lr.best_params_, grid_lr.best_score_

({'C': 1}, np.float64(0.5928889698417444))

In [ ]:
lr = grid_lr.best_estimator_

In [ ]:
print(classification_report(y_train, lr.predict(X_train)))

              precision    recall  f1-score   support

           0       0.98      0.28      0.44     30635
           1       0.05      0.89      0.10      1385

    accuracy                           0.31     32020
   macro avg       0.52      0.58      0.27     32020
weighted avg       0.94      0.31      0.42     32020



In [ ]:
print(classification_report(y_test, lr.predict(X_test)))

              precision    recall  f1-score   support

           0       0.98      0.27      0.43      7659
           1       0.05      0.90      0.10       346

    accuracy                           0.30      8005
   macro avg       0.52      0.58      0.26      8005
weighted avg       0.94      0.30      0.41      8005



# Model B: Random Forest (class_weight='balanced')

In [ ]:
par_grid_rf = {'n_estimators': [100], 'max_depth': [8]}
grid_rf = GridSearchCV(RandomForestClassifier(n_jobs=-1, random_state=123, class_weight='balanced'),
                        param_grid=par_grid_rf, cv=3, scoring='roc_auc', n_jobs=-1)
grid_rf = grid_rf.fit(X_train, y_train)

In [ ]:
grid_rf.best_params_, grid_rf.best_score_

({'max_depth': 8, 'n_estimators': 100}, np.float64(0.6205211662653211))

In [ ]:
rf = grid_rf.best_estimator_

In [ ]:
print(classification_report(y_train, rf.predict(X_train)))
print(classification_report(y_test, rf.predict(X_test)))

              precision    recall  f1-score   support

           0       0.98      0.61      0.75     30635
           1       0.07      0.66      0.13      1385

    accuracy                           0.61     32020
   macro avg       0.52      0.63      0.44     32020
weighted avg       0.94      0.61      0.72     32020

              precision    recall  f1-score   support

           0       0.97      0.60      0.74      7659
           1       0.06      0.60      0.12       346

    accuracy                           0.60      8005
   macro avg       0.52      0.60      0.43      8005
weighted avg       0.93      0.60      0.72      8005



# Model C: XGBoost (scale_pos_weight fixed to imbalance ratio)

In [ ]:
par_grid_xgb = {'n_estimators': [100], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}
grid_xgb = GridSearchCV(XGBClassifier(eval_metric='logloss', random_state=123, scale_pos_weight=imbalance_ratio),
                         param_grid=par_grid_xgb, cv=3, scoring='roc_auc', n_jobs=-1)
grid_xgb = grid_xgb.fit(X_train, y_train)

In [ ]:
grid_xgb.best_params_, grid_xgb.best_score_

({'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100},
 np.float64(0.6361637435342268))

In [ ]:
xgb = grid_xgb.best_estimator_

In [ ]:
print(classification_report(y_train, xgb.predict(X_train)))
print(classification_report(y_test, xgb.predict(X_test)))

              precision    recall  f1-score   support

           0       0.97      0.67      0.79     30635
           1       0.07      0.55      0.12      1385

    accuracy                           0.66     32020
   macro avg       0.52      0.61      0.46     32020
weighted avg       0.93      0.66      0.76     32020

              precision    recall  f1-score   support

           0       0.97      0.67      0.79      7659
           1       0.07      0.55      0.13       346

    accuracy                           0.67      8005
   macro avg       0.52      0.61      0.46      8005
weighted avg       0.93      0.67      0.77      8005



# Model D: LightGBM (class_weight='balanced')

In [ ]:
par_grid_lgb = {'n_estimators': [100], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}
grid_lgb = GridSearchCV(LGBMClassifier(random_state=123, verbose=-1, class_weight='balanced'),
                         param_grid=par_grid_lgb, cv=3, scoring='roc_auc', n_jobs=-1)
grid_lgb = grid_lgb.fit(X_train, y_train)

In [ ]:
grid_lgb.best_params_, grid_lgb.best_score_

({'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100},
 np.float64(0.6368025901361105))

In [ ]:
lgbm = grid_lgb.best_estimator_

In [ ]:
print(classification_report(y_train, lgbm.predict(X_train)))
print(classification_report(y_test, lgbm.predict(X_test)))

              precision    recall  f1-score   support

           0       0.97      0.67      0.79     30635
           1       0.07      0.55      0.12      1385

    accuracy                           0.66     32020
   macro avg       0.52      0.61      0.46     32020
weighted avg       0.93      0.66      0.76     32020

              precision    recall  f1-score   support

           0       0.97      0.67      0.79      7659
           1       0.07      0.55      0.13       346

    accuracy                           0.67      8005
   macro avg       0.52      0.61      0.46      8005
weighted avg       0.93      0.67      0.77      8005



# Compare all models on ROC-AUC

In [ ]:
score = pd.DataFrame([grid_lr.best_score_, grid_rf.best_score_, grid_xgb.best_score_, grid_lgb.best_score_])
name = pd.DataFrame(['logistic_regression', 'random_forest', 'xgboost', 'lightgbm'])

In [ ]:
best_score = pd.concat([name, score], axis=1)
best_score.columns = ['model', 'roc_auc']
best_score.sort_values('roc_auc', ascending=False)

,model,roc_auc
3,lightgbm,0.636803
2,xgboost,0.636164
1,random_forest,0.620521
0,logistic_regression,0.592889


# Threshold tuning on best model

In [ ]:
best_proba = lgbm.predict_proba(X_test)[:, 1]

for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    print(f"for Threshold = {t}")
    pred = np.where(best_proba >= t, 1, 0)
    print(classification_report(y_test, pred))

for Threshold = 0.3
              precision    recall  f1-score   support

           0       0.99      0.12      0.22      7659
           1       0.05      0.97      0.09       346

    accuracy                           0.16      8005
   macro avg       0.52      0.55      0.15      8005
weighted avg       0.95      0.16      0.21      8005

for Threshold = 0.4
              precision    recall  f1-score   support

           0       0.98      0.26      0.41      7659
           1       0.05      0.90      0.10       346

    accuracy                           0.29      8005
   macro avg       0.52      0.58      0.26      8005
weighted avg       0.94      0.29      0.40      8005

for Threshold = 0.5
              precision    recall  f1-score   support

           0       0.97      0.67      0.79      7659
           1       0.07      0.55      0.13       346

    accuracy                           0.67      8005
   macro avg       0.52      0.61      0.46      8005
weighted avg  

#best_threshold = 0.4

In [ ]:
best_threshold = 0.4
final_model = {'model': lgbm, 'threshold': best_threshold}

In [ ]:
joblib.dump(final_model, 'refund_model.pkl')

['refund_model.pkl']

# End of Model 2